# Khipus.ai
## Módulo 5 · Unidad 1
### Tarea 1: construir un generador de publicaciones con Microsoft Foundry

**Nombre del estudiante:** ____________________

<span>© Copyright Notice 2025, Khipus.ai - All Rights Reserved.</span>

Adaptación a Microsoft Foundry: 8 de octubre de 2026. Se conserva la actividad original de generar cinco publicaciones para desarrolladores de IA.

## Consigna y entregables
Construye un generador de **cinco publicaciones en español sobre agentes de IA**. Cada publicación debe incluir un inicio atractivo, una idea técnica útil, al menos tres hashtags, un emoji y una llamada a la acción.

1. Ejecuta y conserva los resultados de los dos prompts incluidos.
2. Evalúa ambas respuestas con la tabla al final del notebook.
3. Redacta una mejora propia del segundo prompt, ejecútala y explica el resultado.
4. Justifica qué salida usarías y qué revisarías antes de publicarla.

El material de contexto es didáctico, no un boletín de novedades. El ejercicio no navega por Internet ni publica en redes sociales. Ejecutar todas las celdas realiza **dos solicitudes de chat**; las repeticiones generan consumo adicional.

## Antes de empezar: preparar el entorno virtual
Estos tres notebooks comparten el único archivo [`requirements.txt`](requirements.txt) de esta carpeta. En PowerShell, desde esta carpeta, crea el entorno si aún no existe y actívalo:

```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
```

Si todavía no instalaste las dependencias (o cambió `requirements.txt`), instálalas una sola vez:

```powershell
python -m pip install -r requirements.txt
```

En VS Code selecciona `.venv` como intérprete y kernel. Si ya preparaste el entorno y los requirements, continúa sin repetir la instalación. Después completa la celda de constantes con la clave y la configuración del recurso.

In [ ]:
# EN CASO DE NO HABER CREADO EL .VENV
# %python -m venv .venv

# ACTIVAR EL ENTORNO VIRTUAL
# Windows
# .venv\Scripts\activate
# macOS / Linux
# source .venv/bin/activate

# INSTALAR LAS DEPENDENCIAS
# %pip install -r requirements.txt

## Configurar la conexión
Define los valores en la celda de constantes y ejecútala antes de crear el cliente:
- `AZURE_OPENAI_API_KEY`: clave del recurso proporcionada por el instructor.
- `AZURE_OPENAI_API_BASE`: endpoint Azure OpenAI terminado en `/openai/v1`.
- `AZURE_OPENAI_CHAT_DEPLOYMENT`: nombre del despliegue que usará esta práctica.

La conexión y las solicitudes toman sus valores de estas constantes. No se leen variables de entorno ni se solicita una contraseña interactiva.

La ruta **Azure OpenAI v1** ya incluye la versión; no se necesita `AZURE_OPENAI_API_VERSION`. El Project Endpoint (`/api/projects/...`) no se utiliza para estas llamadas. La configuración no envía solicitudes; la primera llamada a la API comprobará tus credenciales.

In [ ]:
AZURE_OPENAI_API_KEY = "YOUR_AZURE_API_KEY"
AZURE_OPENAI_API_BASE = "https://khipus-ai-dev-chmrq-resource.openai.azure.com/openai/v1"
AZURE_OPENAI_CHAT_DEPLOYMENT = "gpt-4.1-mini"

In [ ]:
from urllib.parse import urlparse

from openai import OpenAI

base_url = AZURE_OPENAI_API_BASE.strip().rstrip("/") + "/"

url = urlparse(base_url)
if url.scheme != "https" or not url.netloc or url.path != "/openai/v1/":
    raise ValueError("AZURE_OPENAI_API_BASE debe ser el endpoint HTTPS que termina en /openai/v1/.")

if not AZURE_OPENAI_API_KEY.strip():
    raise ValueError("Completa AZURE_OPENAI_API_KEY en la celda de constantes antes de continuar.")

client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY.strip(),
    base_url=base_url,
    timeout=60.0,
    max_retries=2,
)
print("Cliente configurado. La primera solicitud comprobará la conexión.")

## Seleccionar el despliegue
En Azure, `model` recibe el **nombre del despliegue**. Esta práctica usa `gpt-4.1-mini`; puedes repetirla con `gpt-4.1`, también presente en tu proyecto.

Los parámetros de estas celdas están preparados para esos dos despliegues. `gpt-5`, `o4-mini` y `model-router` requieren revisar su compatibilidad antes de sustituirlos. No hace falta desplegar otro modelo para completar la actividad.

In [ ]:
chat_deployment = AZURE_OPENAI_CHAT_DEPLOYMENT.strip()
if not chat_deployment:
    raise ValueError("AZURE_OPENAI_CHAT_DEPLOYMENT no puede estar vacía.")
print("Despliegue de texto:", chat_deployment)

## 1. Definir un contexto común
Ambos prompts usan el mismo material para que puedas comparar el efecto de sus instrucciones. Si deseas hablar de novedades reales, sustituye el contexto por información verificada con sus fuentes y fechas.

In [ ]:
brief = """
Público: desarrolladores de IA que empiezan a construir agentes.
Idioma: español.
Material didáctico:
- Las herramientas permiten al agente solicitar acciones externas mediante interfaces definidas.
- Validar los argumentos de una herramienta ayuda a evitar llamadas mal formadas.
- Un conjunto de casos de prueba permite comparar cambios en un agente.
- Registrar latencia y consumo de tokens ayuda a observar el comportamiento de la aplicación.
- Las acciones sensibles deben tener límites de permisos y supervisión adecuados.
No hay anuncios de productos, fechas de lanzamiento ni cifras de rendimiento en este material.
""".strip()

system_message = (
    "Eres un redactor técnico para desarrolladores de IA. "
    "Usa el contexto proporcionado y no inventes lanzamientos, enlaces, métricas ni fuentes."
)

## 2. Crear una función de generación
Esta función devuelve la respuesta del SDK. Utilizaremos el mismo modelo, temperatura y límite en las dos llamadas para centrar la comparación en el prompt.

In [ ]:
def generate_posts(prompt):
    if not prompt.strip():
        raise ValueError("El prompt no puede estar vacío.")
    result = client.chat.completions.create(
        model=chat_deployment,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": prompt},
        ],
        temperature=0.7,
        max_completion_tokens=1600,
    )
    choice = result.choices[0]
    if not choice.message.content:
        raise RuntimeError(f"Respuesta vacía: finish_reason={choice.finish_reason}.")
    return result

## 3. Ejecutar el prompt inicial
Este prompt describe el tema, pero deja sin especificar varios requisitos de la consigna. Observa qué decide el modelo por su cuenta.

In [ ]:
prompt_v1 = f"Genera cinco publicaciones sobre agentes de IA usando este contexto:\n\n{brief}"
response_v1 = generate_posts(prompt_v1)
print(response_v1.choices[0].message.content)

## 4. Ejecutar un prompt con requisitos explícitos
La tarea del estudiante es evaluar el cumplimiento; no se debe asumir que una respuesta cumple todas las restricciones solo porque se solicitaron.

In [ ]:
prompt_v2 = f"""
Genera exactamente cinco publicaciones en español para desarrolladores que empiezan
a construir agentes de IA. Usa solamente el contexto incluido más abajo.

Requisitos de cada publicación:
- Empieza con un titular o frase breve que capte la atención.
- Explica una idea técnica distinta del contexto y ofrece una recomendación aplicable.
- Incluye al menos tres hashtags relevantes y exactamente un emoji.
- Termina con una llamada a la acción para comentar, probar o compartir experiencias.
- Mantén entre 40 y 70 palabras, contando hashtags.
- No inventes URLs, fuentes, lanzamientos, fechas ni resultados medidos.

Formato: cinco elementos numerados del 1 al 5, sin introducción ni conclusión adicional.

Contexto:
{brief}
""".strip()

response_v2 = generate_posts(prompt_v2)
print(response_v2.choices[0].message.content)

## 5. Revisar finalización y consumo
Si una respuesta está truncada o filtrada, regístralo antes de evaluar su calidad. Los tokens totales permiten comparar consumo, no determinar automáticamente cuál texto es mejor.

In [ ]:
for label, result in [("Prompt inicial", response_v1), ("Prompt detallado", response_v2)]:
    print(label)
    print("  Motivo de finalización:", result.choices[0].finish_reason)
    if result.choices[0].finish_reason == "length":
        print("  La salida alcanzó el límite: revisa su integridad antes de evaluarla.")
    if result.usage is not None:
        print("  Tokens de entrada:", result.usage.prompt_tokens)
        print("  Tokens de salida:", result.usage.completion_tokens)
        print("  Tokens totales:", result.usage.total_tokens)

## 6. Cómo completar la evaluación (guía paso a paso)
**Paso 1 – Ejecuta y revisa.** Corre las secciones 3, 4 y 5. Lee con calma `response_v1` y `response_v2`; no te fíes de que "se pidió" algo: comprueba que realmente aparece.

**Paso 2 – Mide con la celda de ayuda.** La celda siguiente cuenta por ti publicaciones, palabras, hashtags y emojis. Úsala con cada respuesta (`check_posts(response_v1)`, `check_posts(response_v2)` y, más adelante, `check_posts(response_v3)`). Lo que no se puede medir automáticamente (inicio atractivo, idea técnica distinta, llamada a la acción, datos inventados) debes juzgarlo leyendo el texto.

**Paso 3 – Rellena la tabla de la sección 7.** Una fila por criterio y una columna por prompt:
- Escribe **Sí** si **todas** las publicaciones cumplen, **No** si ninguna cumple, **Parcial** si solo algunas (indica cuáles en Evidencia).
- En filas con números (palabras, tokens), escribe el valor real, por ejemplo `52, 48, 75, 61, 40`.
- `finish_reason` y tokens salen de la sección 5. Si es `length`, la respuesta quedó cortada: anótalo.
- En **Evidencia / observaciones** cita brevemente de dónde sale tu juicio (p. ej. "pub. 3 tiene 2 emojis", "v1 no usa hashtags").
- **Puntaje global**: de 1 (muy deficiente) a 5 (cumple todo con buena calidad), con base en lo anterior.

**Paso 4 – Crea tu prompt v3 (sección 8).** Parte de `prompt_v2`, elige **un problema concreto** que viste en la tabla y corrígelo (por ejemplo: pedir un formato fijo `Titular / Idea / Hashtags / CTA`, añadir un ejemplo de una publicación, o endurecer el límite de palabras). Ejecútalo y rellena la columna v3.

**Paso 5 – Escribe las conclusiones (sección 9).** Responde en 1 a 3 frases cada pregunta, apoyándote en la evidencia de tu tabla.

**Ejemplo de fila completada:**

| Criterio | v1 | v2 | v3 | Evidencia / observaciones |
|---|---|---|---|---|
| Exactamente un emoji | No | Parcial | Sí | v1 no usa emojis; en v2 la pub. 4 tiene 2; en v3 cada una tiene 1 |

In [ ]:
import re

EMOJI_RE = re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF\u2B00-\u2BFF\uFE0F]")

def check_posts(result):
    """Muestra conteos por publicación para ayudarte a completar la tabla."""
    text = result.choices[0].message.content
    posts = [p.strip() for p in re.split(r"(?m)^\s*\d+[\.\)]\s+", text) if p.strip()]
    print(f"Publicaciones detectadas: {len(posts)}")
    for i, post in enumerate(posts, 1):
        words = len(post.split())
        hashtags = len(re.findall(r"#\w+", post))
        emojis = len(EMOJI_RE.findall(post.replace("\uFE0F", "")))
        print(f"  {i}. palabras={words} (40-70: {'sí' if 40 <= words <= 70 else 'no'}), hashtags={hashtags}, emojis={emojis}")
    print("Finalización:", result.choices[0].finish_reason, "| tokens totales:", result.usage.total_tokens if result.usage else "n/d")

check_posts(response_v1)
check_posts(response_v2)

## 7. Tabla de evaluación
Rellena la tabla con tus hallazgos (sigue la guía de la sección 6) tras leer cada salida. Escribe **Sí** si cumple, **No** si no cumple o **Parcial** si cumple solo en parte, y anota la evidencia (por ejemplo, la publicación o el conteo de palabras que lo demuestra).

| Criterio | Prompt inicial (v1) | Prompt detallado (v2) | Tu prompt mejorado (v3) | Evidencia / observaciones |
|---|---|---|---|---|
| Genera exactamente cinco publicaciones | | | | |
| Está en español | | | | |
| Inicio atractivo (titular o frase breve) | | | | |
| Idea técnica útil, distinta en cada publicación | | | | |
| Al menos tres hashtags relevantes | | | | |
| Exactamente un emoji | | | | |
| Llamada a la acción | | | | |
| Longitud de 40 a 70 palabras por publicación | | | | |
| Usa solo el contexto (sin URLs, fuentes, fechas ni métricas inventadas) | | | | |
| Formato numerado, sin introducción ni conclusión extra | | | | |
| Motivo de finalización (`finish_reason`) | | | | |
| Tokens totales | | | | |
| **Puntaje global (1 a 5)** | | | | |

## 8. Mejora propia del segundo prompt
Escribe y ejecuta tu versión mejorada (`prompt_v3`) en la celda siguiente. Luego ejecuta `check_posts(response_v3)` (descomenta las dos últimas líneas de la celda y añade esa llamada) y completa la columna v3 de la tabla de la sección 7.

In [ ]:
# Escribe aquí tu mejora del prompt_v2
prompt_v3 = f"""

Contexto:
{brief}
""".strip()

# response_v3 = generate_posts(prompt_v3)
# print(response_v3.choices[0].message.content)

## 9. Conclusiones
Responde con tus propias palabras.

| Pregunta | Tu respuesta |
|---|---|
| ¿Qué cambió entre v1 y v2 y qué efecto tuvo en el resultado? | |
| ¿Qué cambiaste en v3 y por qué? ¿Mejoró el resultado? | |
| ¿Qué restricciones no se cumplieron aunque se pidieron explícitamente? | |
| ¿Qué salida usarías para publicar y por qué? | |
| ¿Qué revisarías manualmente antes de publicarla? | |